In [1]:
import optuna
import axelrod
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def sigmoid(x, center, scale=10):
        return 1 / (1 + exp(-scale * (x - center)))
    
    @staticmethod
    def fuzzy_gate(mu_D, d_thresh, mu_C, c_thresh):
        d_condition = FuzzyMethods.sigmoid(mu_D, center=d_thresh)
        c_condition = 1 - FuzzyMethods.sigmoid(mu_C, center=c_thresh)

        w1 = d_condition * c_condition
        w2 = 1 - w1

        z1 = 1
        z2 = 0

        z = (w1 * z1 + w2 * z2) / (w1 + w2 + 1e-6)
        return z
    
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [3]:
def build_fuzzy_player(params):
    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')
    

    _cooperation.automf(names=["low", "medium", "high"])
    _adaptivity.automf(names=["no", "yes"])
    _forgiveness.automf(names=["low", "medium", "high"])
    _forgiveness['low'] = fuzz.gaussmf(_forgiveness.universe, 0, 25)
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [25, 50, 75])
    _stochastic.automf(names=["none", "sometimes", "always"])

    # Resulting strategy MFs — also being optimized
    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [
        0,
        params['D_b'],
        params['D_c']
    ])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [
        params['C_a'],
        params['C_b'],
        100
    ])

    # Rebuild rules using the fresh variables above
    rule1 = ctrl.Rule(
        _cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']),
        _resulting_strategy['D']
    )
    rule2 = ctrl.Rule(
        _forgiveness['low'] & _cooperation['high'],
        _resulting_strategy['C']
    )
    rule3 = ctrl.Rule(
        _stochastic['always'] | _adaptivity['no'],
        _resulting_strategy['D']
    )
    rule4 = ctrl.Rule(
        _cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']),
        _resulting_strategy['D']
    )
    rule5 = ctrl.Rule(
        _cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'],
        _resulting_strategy['C']
    )

    strategy_ctrl  = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
    _chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

    # Build the player class dynamically, capturing everything in closure
    class OptimizedFuzzy(Player):

        # Override class-level FIS components with the fresh ones
        cooperation = _cooperation
        adaptivity = _adaptivity
        stochastic = _stochastic
        forgiveness = _forgiveness
        resulting_strategy = _resulting_strategy
        chosen_strategy = _chosen_strategy

        d_thresh = params['d_threshold']
        c_thresh = params['c_threshold']

        # Reset state so trials don't bleed into each other
        first_time = True
        h = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: axelrod.Player) -> Action:

            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val = self.chosen_strategy.output['resulting_strategy']
            except KeyError:
                # No rules fired — default to cooperate
                return C
            except Exception:
                return C

            d_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['D'].mf,
                output_val
            )
            c_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['C'].mf,
                output_val
            )

            output = FuzzyMethods.fuzzy_gate(d_membership, self.d_thresh, c_membership, self.c_thresh)

            if(output > 0.5):
                return D
            
            return C

    return OptimizedFuzzy()

In [4]:

def objective(trial):

    D_a = 0
    D_b = trial.suggest_int('D_b', 1, 49)
    D_c = trial.suggest_int('D_c', D_b, 60)

    C_a = trial.suggest_int('C_a', 25, 74)
    C_b = trial.suggest_int('C_b', C_a, 99)
    C_c = 99

    # --- THRESHOLDS ---
    d_threshold = trial.suggest_float('d_threshold', 0.1, 0.99)
    c_threshold = trial.suggest_float('c_threshold', 0.2, 0.9)

    params = {
    'D_a': D_a, 'D_b': D_b, 'D_c': D_c,
    'C_a': C_a, 'C_b': C_b, 'C_c': C_c,
    'd_threshold': d_threshold,
    'c_threshold': c_threshold,
    }

    try:
        fuzzy_player = build_fuzzy_player(params)
    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

    opponents = [player() for player in axelrod.stewart_plotkin_strategies]

    try:
        tournament = axelrod.Tournament(
            [fuzzy_player] + opponents,
            turns=200,
            repetitions = 5
        )
        results = tournament.play(progress_bar=False)
    except Exception as e:
        print(f"Tournament failed: {e}")
        return 0.0

    return np.mean(results.normalised_scores[0])

In [5]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_full_start_c_d_fixed_sugeno',
    storage='sqlite:///fuzzy_optuna_full_start_c_d_fixed.db_sugeno',
    load_if_exists=True
)
study.enqueue_trial({
        'D_b': 25, 'D_c': 50,
        'C_a': 35, 'C_b': 75,
        'd_threshold': 0.4,
        'c_threshold': 0.6,
})
study.optimize(objective, n_trials=300, show_progress_bar=True)

print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")

[I 2026-03-05 16:48:30,160] A new study created in RDB with name: fuzzy_optimization_full_start_c_d_fixed_sugeno
Best trial: 0. Best value: 2.75743:   0%|          | 1/300 [00:17<1:28:58, 17.86s/it]

[I 2026-03-05 16:48:48,058] Trial 0 finished with value: 2.7574285714285716 and parameters: {'D_b': 25, 'D_c': 50, 'C_a': 35, 'C_b': 75, 'd_threshold': 0.4, 'c_threshold': 0.6}. Best is trial 0 with value: 2.7574285714285716.


Best trial: 0. Best value: 2.75743:   1%|          | 2/300 [00:33<1:22:47, 16.67s/it]

[I 2026-03-05 16:49:03,913] Trial 1 finished with value: 2.7365000000000004 and parameters: {'D_b': 15, 'D_c': 24, 'C_a': 61, 'C_b': 77, 'd_threshold': 0.20973130204163276, 'c_threshold': 0.2079604803311533}. Best is trial 0 with value: 2.7574285714285716.


Best trial: 0. Best value: 2.75743:   1%|          | 3/300 [00:47<1:14:57, 15.14s/it]

...

[I 2026-03-05 18:55:51,007] Trial 297 finished with value: 2.733642857142857 and parameters: {'D_b': 25, 'D_c': 26, 'C_a': 45, 'C_b': 78, 'd_threshold': 0.41195201606772863, 'c_threshold': 0.22806293229258107}. Best is trial 75 with value: 2.7854285714285716.


Best trial: 75. Best value: 2.78543: 100%|█████████▉| 299/300 [2:07:34<00:20, 20.86s/it]

[I 2026-03-05 18:56:04,243] Trial 298 finished with value: 2.7036428571428575 and parameters: {'D_b': 24, 'D_c': 43, 'C_a': 39, 'C_b': 79, 'd_threshold': 0.2946750996414744, 'c_threshold': 0.5684966871987582}. Best is trial 75 with value: 2.7854285714285716.


Best trial: 75. Best value: 2.78543: 100%|██████████| 300/300 [2:07:46<00:00, 25.55s/it]


[I 2026-03-05 18:56:16,301] Trial 299 finished with value: 2.6740000000000004 and parameters: {'D_b': 22, 'D_c': 33, 'C_a': 45, 'C_b': 75, 'd_threshold': 0.10002744453768452, 'c_threshold': 0.6057925646953585}. Best is trial 75 with value: 2.7854285714285716.

=== OPTIMIZATION COMPLETE ===
Best score:  2.7854
Best params: {'D_b': 26, 'D_c': 28, 'C_a': 41, 'C_b': 66, 'd_threshold': 0.326854282048886, 'c_threshold': 0.5201250422764233}

=== PARAMETER IMPORTANCE ===
  d_threshold: 0.4377
  D_b: 0.3931
  c_threshold: 0.1207
  C_a: 0.0485
